In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

# Definir el esquema exacto para la tabla de metadatos Bronze
schema_bronze = StructType([
    StructField("document_id", StringType(), False),
    StructField("file_name", StringType(), False),
    StructField("file_path", StringType(), False),
    StructField("file_type", StringType(), True),
    StructField("category", StringType(), False),
    StructField("file_size", LongType(), True),
    StructField("ingestion_date", TimestampType(), False),
    StructField("processing_status", StringType(), False),
    StructField("error_message", StringType(), True)
])

# Crear la tabla Delta vacía en el Lakehouse si no existe
spark.createDataFrame([], schema_bronze).write.format("delta").mode("ignore").saveAsTable("bronze_documents")

print("Tabla 'bronze_documents' inicializada correctamente.")

In [ ]:
import os
import hashlib
from datetime import datetime
from pyspark.sql import Row
from delta.tables import DeltaTable

# ---------------------------------------------------------
# CONFIGURACIÓN Y REGLAS DE NEGOCIO (FASE 3)
# ---------------------------------------------------------
bronze_path = "Files/Bronze/Documents"
categorias_validas = ["legal", "hr", "technical", "esg", "financial", "security", "operations"]
formatos_soportados = ["pdf", "txt", "md"]

# 1. Listar archivos en OneLake
try:
    files_in_lakehouse = mssparkutils.fs.ls(bronze_path)
    print(f"Detectados {len(files_in_lakehouse)} archivos en OneLake.")
except Exception as e:
    raise Exception(f"Fallo crítico al acceder a {bronze_path}: {str(e)}")

rows_to_insert = []

# 2. Procesar cada archivo para validar y estructurar metadatos
for f in files_in_lakehouse:
    file_name = f.name
    file_path = f.path
    file_size = f.size
    
    # Extraer extensión
    file_type = os.path.splitext(file_name)[1].replace(".", "").lower()
    
    # --- CLASIFICACIÓN DE CATEGORÍA ---
    parts = file_name.split("_")
    prefix = parts[0].lower()
    category = prefix if prefix in categorias_validas else "operations"
    
    # Generar ID único idempotente
    document_id = hashlib.md5(file_name.encode('utf-8')).hexdigest()
    
    # --- REGLA DE VALIDACIÓN Y ESTADO INICIAL ---
    if file_type in formatos_soportados:
        processing_status = "pending"
        error_message = None
    else:
        processing_status = "skipped"
        error_message = f"Formato '{file_type}' no soportado."
    
    metadata_row = Row(
        document_id=document_id,
        file_name=file_name,
        file_path=file_path,
        file_type=file_type,
        category=category,
        file_size=file_size,
        ingestion_date=datetime.now(),
        processing_status=processing_status,
        error_message=error_message
    )
    rows_to_insert.append(metadata_row)

# ---------------------------------------------------------
# MERGE IDEMPOTENTE USANDO EL CATÁLOGO (DeltaTable.forName)
# ---------------------------------------------------------
if rows_to_insert:
    # Obtener el esquema de la tabla desde la metabase de Fabric
    schema_bronze = spark.table("bronze_documents").schema
    df_new_metadata = spark.createDataFrame(rows_to_insert, schema=schema_bronze)
    
    print("Sincronizando y aplicando MERGE en la tabla Delta 'bronze_documents'...")
    
    # FORMA CORRECTA EN FABRIC: Referenciar la tabla por su nombre de catálogo
    delta_target = DeltaTable.forName(spark, "bronze_documents")

    
   # Ejecutar MERGE: Inserta nuevos y actualiza si el tamaño ha cambiado
    delta_target.alias("target") \
        .merge(
            df_new_metadata.alias("source"),
            "target.document_id = source.document_id"
        ) \
        .whenMatchedUpdate(
            # Si el archivo tiene el mismo nombre pero ha cambiado de tamaño
            condition = "target.file_size != source.file_size",
            set = {
                "file_size": "source.file_size",
                "ingestion_date": "source.ingestion_date",
                "processing_status": "'pending'",  # Reiniciamos a pending para reprocesarlo
                "error_message": "source.error_message"
            }
        ) \
        .whenNotMatchedInsertAll() \
        .execute()
    
    print("¡Sincronización completada con éxito!")
else:
    print("No se encontraron archivos para procesar.")

StatementMeta(, 86efa4e1-4ee0-4c56-b2d8-32f41e7e9712, 3, Finished, Available, Finished, False)

Detectados 8 archivos en OneLake.
Sincronizando y aplicando MERGE en la tabla Delta 'bronze_documents'...
¡Sincronización completada con éxito!


In [2]:
# Comprobamos el número total de registros (debería ser exactamente 8: 6 iniciales + 2 nuevos)
total_registros = spark.sql("SELECT COUNT(*) as total FROM Bronze_Documents").collect()[0]["total"]
print(f"Número total de registros en la tabla: {total_registros} (Esperado: 8)")

# Visualizamos la tabla ordenada para inspeccionar las categorías 'financial' y 'operations'
spark.sql("""
    SELECT file_name, category, file_type, processing_status, ingestion_date 
    FROM Bronze_Documents 
    ORDER BY ingestion_date DESC
""").show(truncate=False)

StatementMeta(, 86efa4e1-4ee0-4c56-b2d8-32f41e7e9712, 4, Finished, Available, Finished, True)

Número total de registros en la tabla: 8 (Esperado: 8)
+----------------------------------------------------------+----------+---------+-----------------+--------------------------+
|file_name                                                 |category  |file_type|processing_status|ingestion_date            |
+----------------------------------------------------------+----------+---------+-----------------+--------------------------+
|financial_presupuesto_anual_2026.txt                      |financial |txt      |pending          |2026-08-07 10:46:22.005213|
|esg_IB_Informe_Sostenibilidad.pdf                         |esg       |pdf      |pending          |2026-08-07 10:46:22.00489 |
|technical_Guia_Profesional_Tramitacion_autoconsumo_v.6.pdf|technical |pdf      |processed        |2026-08-07 10:40:06.024192|
|security_manual_seguridad_plantas_solares.txt             |security  |txt      |processed        |2026-08-07 10:40:06.024003|
|operations_procedimiento_compras_proveedores.txt       